# HW 3

---  

**นาย ปิยพงษ์ อาจศึก**
**B6726218**

In [ ]:
from search import *
from notebook import heatmap, gaussian_kernel, display_visual, plot_NQueens

# Needed to hide warnings in the matplotlib sections
import warnings
warnings.filterwarnings("ignore")

In [3]:
%matplotlib inline
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib import lines

from ipywidgets import interact
import ipywidgets as widgets
from IPython.display import display
import time

In [4]:
romania_map = UndirectedGraph(dict(
    Arad=dict(Zerind=75, Sibiu=140, Timisoara=118),
    Bucharest=dict(Urziceni=85, Pitesti=101, Giurgiu=90, Fagaras=211),
    Craiova=dict(Drobeta=120, Rimnicu=146, Pitesti=138),
    Drobeta=dict(Mehadia=75),
    Eforie=dict(Hirsova=86),
    Fagaras=dict(Sibiu=99),
    Hirsova=dict(Urziceni=98),
    Iasi=dict(Vaslui=92, Neamt=87),
    Lugoj=dict(Timisoara=111, Mehadia=70),
    Oradea=dict(Zerind=71, Sibiu=151),
    Pitesti=dict(Rimnicu=97),
    Rimnicu=dict(Sibiu=80),
    Urziceni=dict(Vaslui=142)))

romania_map.locations = dict(
    Arad=(91, 492), Bucharest=(400, 327), Craiova=(253, 288),
    Drobeta=(165, 299), Eforie=(562, 293), Fagaras=(305, 449),
    Giurgiu=(375, 270), Hirsova=(534, 350), Iasi=(473, 506),
    Lugoj=(165, 379), Mehadia=(168, 339), Neamt=(406, 537),
    Oradea=(131, 571), Pitesti=(320, 368), Rimnicu=(233, 410),
    Sibiu=(207, 457), Timisoara=(94, 410), Urziceni=(456, 350),
    Vaslui=(509, 444), Zerind=(108, 531))

In [5]:
# node colors, node positions and node label positions
node_colors = { node: 'white' for node in romania_map.locations.keys() }
node_positions = romania_map.locations
node_label_pos = { k: [ v[0], v[1] - 10 ]  for k,v in romania_map.locations.items() }
edge_weights = {(k, k2) : v2 for k, v in romania_map.graph_dict.items() for k2, v2 in v.items()}

romania_graph_data = {  
    'graph_dict' : romania_map.graph_dict,
    'node_colors': node_colors,
    'node_positions': node_positions,
    'node_label_positions': node_label_pos,
    'edge_weights': edge_weights
}

### Algorithm

In [6]:
def tree_breadth_search_for_vis(problem):
    """Search through the successors of a problem to find a goal.
    The argument frontier should be an empty queue.
    Don't worry about repeated paths to a state. [Figure 3.7]"""
    
    # we use these two variables at the time of visualisations
    iterations = 0
    all_node_colors = []
    node_colors = {k : 'white' for k in problem.graph.nodes()}
    
    #Adding first node to the queue
    frontier = deque([Node(problem.initial)])
    
    node_colors[Node(problem.initial).state] = "orange"
    iterations += 1
    all_node_colors.append(dict(node_colors))
    
    while frontier:
        #Popping first node of queue
        node = frontier.popleft()
        
        # modify the currently searching node to red
        node_colors[node.state] = "red"
        iterations += 1
        all_node_colors.append(dict(node_colors))
        
        if problem.goal_test(node.state):
            # modify goal node to green after reaching the goal
            node_colors[node.state] = "green"
            iterations += 1
            all_node_colors.append(dict(node_colors))
            return(iterations, all_node_colors, node)
        
        frontier.extend(node.expand(problem))
           
        for n in node.expand(problem):
            node_colors[n.state] = "orange"
            iterations += 1
            all_node_colors.append(dict(node_colors))

        # modify the color of explored nodes to gray
        node_colors[node.state] = "gray"
        iterations += 1
        all_node_colors.append(dict(node_colors))
        
    return None

def breadth_first_tree_search(problem):
    "Search the shallowest nodes in the search tree first."
    iterations, all_node_colors, node = tree_breadth_search_for_vis(problem)
    return(iterations, all_node_colors, node)

In [7]:
def tree_depth_search_for_vis(problem):
    """Search through the successors of a problem to find a goal.
    The argument frontier should be an empty queue.
    Don't worry about repeated paths to a state. [Figure 3.7]"""
    
    # we use these two variables at the time of visualisations
    iterations = 0
    all_node_colors = []
    node_colors = {k : 'white' for k in problem.graph.nodes()}
    
    #Adding first node to the stack
    frontier = [Node(problem.initial)]
    
    node_colors[Node(problem.initial).state] = "orange"
    iterations += 1
    all_node_colors.append(dict(node_colors))
    
    while frontier:
        #Popping first node of stack
        node = frontier.pop()
        
        # modify the currently searching node to red
        node_colors[node.state] = "red"
        iterations += 1
        all_node_colors.append(dict(node_colors))
        
        if problem.goal_test(node.state):
            # modify goal node to green after reaching the goal
            node_colors[node.state] = "green"
            iterations += 1
            all_node_colors.append(dict(node_colors))
            return(iterations, all_node_colors, node)
        
        frontier.extend(node.expand(problem))
           
        for n in node.expand(problem):
            node_colors[n.state] = "orange"
            iterations += 1
            all_node_colors.append(dict(node_colors))

        # modify the color of explored nodes to gray
        node_colors[node.state] = "gray"
        iterations += 1
        all_node_colors.append(dict(node_colors))
        
    return None

def depth_first_tree_search(problem):
    "Search the deepest nodes in the search tree first."
    iterations, all_node_colors, node = tree_depth_search_for_vis(problem)
    return(iterations, all_node_colors, node)

In [8]:
def breadth_first_search_graph(problem):
    "[Figure 3.11]"
    
    # we use these two variables at the time of visualisations
    iterations = 0
    all_node_colors = []
    node_colors = {k : 'white' for k in problem.graph.nodes()}
    
    node = Node(problem.initial)
    
    node_colors[node.state] = "red"
    iterations += 1
    all_node_colors.append(dict(node_colors))
      
    if problem.goal_test(node.state):
        node_colors[node.state] = "green"
        iterations += 1
        all_node_colors.append(dict(node_colors))
        return(iterations, all_node_colors, node)
    
    frontier = deque([node])
    
    # modify the color of frontier nodes to blue
    node_colors[node.state] = "orange"
    iterations += 1
    all_node_colors.append(dict(node_colors))
        
    explored = set()
    while frontier:
        node = frontier.popleft()
        node_colors[node.state] = "red"
        iterations += 1
        all_node_colors.append(dict(node_colors))
        
        explored.add(node.state)     
        
        for child in node.expand(problem):
            if child.state not in explored and child not in frontier:
                if problem.goal_test(child.state):
                    node_colors[child.state] = "green"
                    iterations += 1
                    all_node_colors.append(dict(node_colors))
                    return(iterations, all_node_colors, child)
                frontier.append(child)

                node_colors[child.state] = "orange"
                iterations += 1
                all_node_colors.append(dict(node_colors))
                    
        node_colors[node.state] = "gray"
        iterations += 1
        all_node_colors.append(dict(node_colors))
    return None

In [9]:
def graph_search_for_vis(problem):
    """Search through the successors of a problem to find a goal.
    The argument frontier should be an empty queue.
    If two paths reach a state, only use the first one. [Figure 3.7]"""
    # we use these two variables at the time of visualisations
    iterations = 0
    all_node_colors = []
    node_colors = {k : 'white' for k in problem.graph.nodes()}
    
    frontier = [(Node(problem.initial))]
    explored = set()
    
    # modify the color of frontier nodes to orange
    node_colors[Node(problem.initial).state] = "orange"
    iterations += 1
    all_node_colors.append(dict(node_colors))
      
    while frontier:
        # Popping first node of stack
        node = frontier.pop()
        
        # modify the currently searching node to red
        node_colors[node.state] = "red"
        iterations += 1
        all_node_colors.append(dict(node_colors))
        
        if problem.goal_test(node.state):
            # modify goal node to green after reaching the goal
            node_colors[node.state] = "green"
            iterations += 1
            all_node_colors.append(dict(node_colors))
            return(iterations, all_node_colors, node)
        
        explored.add(node.state)
        frontier.extend(child for child in node.expand(problem)
                        if child.state not in explored and
                        child not in frontier)
        
        for n in frontier:
            # modify the color of frontier nodes to orange
            node_colors[n.state] = "orange"
            iterations += 1
            all_node_colors.append(dict(node_colors))

        # modify the color of explored nodes to gray
        node_colors[node.state] = "gray"
        iterations += 1
        all_node_colors.append(dict(node_colors))
        
    return None


def depth_first_graph_search(problem):
    """Search the deepest nodes in the search tree first."""
    iterations, all_node_colors, node = graph_search_for_vis(problem)
    return(iterations, all_node_colors, node)

In [10]:
def best_first_graph_search_for_vis(problem, f):
    """Search the nodes with the lowest f scores first.
    You specify the function f(node) that you want to minimize; for example,
    if f is a heuristic estimate to the goal, then we have greedy best
    first search; if f is node.depth then we have breadth-first search.
    There is a subtlety: the line "f = memoize(f, 'f')" means that the f
    values will be cached on the nodes as they are computed. So after doing
    a best first search you can examine the f values of the path returned."""
    
    # we use these two variables at the time of visualisations
    iterations = 0
    all_node_colors = []
    node_colors = {k : 'white' for k in problem.graph.nodes()}
    
    f = memoize(f, 'f')
    node = Node(problem.initial)
    
    node_colors[node.state] = "red"
    iterations += 1
    all_node_colors.append(dict(node_colors))
    
    if problem.goal_test(node.state):
        node_colors[node.state] = "green"
        iterations += 1
        all_node_colors.append(dict(node_colors))
        return(iterations, all_node_colors, node)
    
    frontier = PriorityQueue('min', f)
    frontier.append(node)
    
    node_colors[node.state] = "orange"
    iterations += 1
    all_node_colors.append(dict(node_colors))
    
    explored = set()
    while frontier:
        node = frontier.pop()
        
        node_colors[node.state] = "red"
        iterations += 1
        all_node_colors.append(dict(node_colors))
        
        if problem.goal_test(node.state):
            node_colors[node.state] = "green"
            iterations += 1
            all_node_colors.append(dict(node_colors))
            return(iterations, all_node_colors, node)
        
        explored.add(node.state)
        for child in node.expand(problem):
            if child.state not in explored and child not in frontier:
                frontier.append(child)
                node_colors[child.state] = "orange"
                iterations += 1
                all_node_colors.append(dict(node_colors))
            elif child in frontier:
                incumbent = frontier[child]
                if f(child) < incumbent:
                    del frontier[child]
                    frontier.append(child)
                    node_colors[child.state] = "orange"
                    iterations += 1
                    all_node_colors.append(dict(node_colors))

        node_colors[node.state] = "gray"
        iterations += 1
        all_node_colors.append(dict(node_colors))
    return None

In [11]:
def uniform_cost_search_graph(problem):
    "[Figure 3.14]"
    #Uniform Cost Search uses Best First Search algorithm with f(n) = g(n)
    iterations, all_node_colors, node = best_first_graph_search_for_vis(problem, lambda node: node.path_cost)
    return(iterations, all_node_colors, node)


In [12]:
def depth_limited_search_graph(problem, limit = -1):
    '''
    Perform depth first search of graph g.
    if limit >= 0, that is the maximum depth of the search.
    '''
    # we use these two variables at the time of visualisations
    iterations = 0
    all_node_colors = []
    node_colors = {k : 'white' for k in problem.graph.nodes()}
    
    frontier = [Node(problem.initial)]
    explored = set()
    
    cutoff_occurred = False
    node_colors[Node(problem.initial).state] = "orange"
    iterations += 1
    all_node_colors.append(dict(node_colors))
      
    while frontier:
        # Popping first node of queue
        node = frontier.pop()
        
        # modify the currently searching node to red
        node_colors[node.state] = "red"
        iterations += 1
        all_node_colors.append(dict(node_colors))
        
        if problem.goal_test(node.state):
            # modify goal node to green after reaching the goal
            node_colors[node.state] = "green"
            iterations += 1
            all_node_colors.append(dict(node_colors))
            return(iterations, all_node_colors, node)

        elif limit >= 0:
            cutoff_occurred = True
            limit += 1
            all_node_colors.pop()
            iterations -= 1
            node_colors[node.state] = "gray"

        
        explored.add(node.state)
        frontier.extend(child for child in node.expand(problem)
                        if child.state not in explored and
                        child not in frontier)
        
        for n in frontier:
            limit -= 1
            # modify the color of frontier nodes to orange
            node_colors[n.state] = "orange"
            iterations += 1
            all_node_colors.append(dict(node_colors))

        # modify the color of explored nodes to gray
        node_colors[node.state] = "gray"
        iterations += 1
        all_node_colors.append(dict(node_colors))
        
    return 'cutoff' if cutoff_occurred else None


def depth_limited_search_for_vis(problem):
    """Search the deepest nodes in the search tree first."""
    iterations, all_node_colors, node = depth_limited_search_graph(problem)
    return(iterations, all_node_colors, node)     

In [13]:
def iterative_deepening_search_for_vis(problem):
    for _ in range(sys.maxsize):
        iterations, all_node_colors, node=depth_limited_search_for_vis(problem)
        if iterations:
            return (iterations, all_node_colors, node)

In [14]:
def greedy_best_first_search(problem, h=None):
    """Greedy Best-first graph search is an informative searching algorithm with f(n) = h(n).
    You need to specify the h function when you call best_first_search, or
    else in your Problem subclass."""
    h = memoize(h or problem.h, 'h')
    iterations, all_node_colors, node = best_first_graph_search_for_vis(problem, lambda n: h(n))
    return(iterations, all_node_colors, node)


In [15]:
def astar_search_graph(problem, h=None):
    """A* search is best-first graph search with f(n) = g(n)+h(n).
    You need to specify the h function when you call astar_search, or
    else in your Problem subclass."""
    h = memoize(h or problem.h, 'h')
    iterations, all_node_colors, node = best_first_graph_search_for_vis(
        problem, 
        lambda n: n.path_cost + h(n)
    )
    return(iterations, all_node_colors, node)


In [20]:
infinity = float('inf')

def recursive_best_first_search_for_vis(problem, h=None):
    """[Figure 3.26] Recursive best-first search"""
    # we use these two variables at the time of visualizations
    iterations = 0
    all_node_colors = []
    node_colors = {k : 'white' for k in problem.graph.nodes()}
    
    h = memoize(h or problem.h, 'h')
    
    def RBFS(problem, node, flimit):
        nonlocal iterations
        def color_city_and_update_map(node, color):
            node_colors[node.state] = color
            nonlocal iterations
            iterations += 1
            all_node_colors.append(dict(node_colors))
            
        if problem.goal_test(node.state):
            color_city_and_update_map(node, 'green')
            return (iterations, all_node_colors, node), 0  # the second value is immaterial
        
        successors = node.expand(problem)
        if len(successors) == 0:
            color_city_and_update_map(node, 'gray')
            return (iterations, all_node_colors, None), infinity
        
        for s in successors:
            color_city_and_update_map(s, 'orange')
            s.f = max(s.path_cost + h(s), node.f)
            
        while True:
            # Order by lowest f value
            successors.sort(key=lambda x: x.f)
            best = successors[0]
            if best.f > flimit:
                color_city_and_update_map(node, 'gray')
                return (iterations, all_node_colors, None), best.f
            
            if len(successors) > 1:
                alternative = successors[1].f
            else:
                alternative = infinity
                
            node_colors[node.state] = 'gray'
            node_colors[best.state] = 'red'
            iterations += 1
            all_node_colors.append(dict(node_colors))
            result, best.f = RBFS(problem, best, min(flimit, alternative))
            if result[2] is not None:
                color_city_and_update_map(node, 'green')
                return result, best.f
            else:
                color_city_and_update_map(node, 'red')
                
    node = Node(problem.initial)
    node.f = h(node)
    
    node_colors[node.state] = 'red'
    iterations += 1
    all_node_colors.append(dict(node_colors))
    result, bestf = RBFS(problem, node, infinity)
    return result

In [18]:
all_node_colors = []

algorithms = {
    "Breadth First Tree Search": tree_breadth_search_for_vis,
    "Depth First Tree Search": tree_depth_search_for_vis,
    "Breadth First Search": breadth_first_search_graph,
    "Depth First Graph Search": graph_search_for_vis,
    "Best First Graph Search": best_first_graph_search_for_vis,
    "Uniform Cost Search": uniform_cost_search_graph,
    "Depth Limited Search": depth_limited_search_for_vis,
    "Iterative Deepening Search": iterative_deepening_search_for_vis,
    "Greedy Best First Search": greedy_best_first_search,
    "A-star Search": astar_search_graph,
    "Recursive Best First Search": recursive_best_first_search_for_vis
}

display_visual(romania_graph_data, algorithm=algorithms, user_input=True)

Dropdown(description='Search algorithm: ', index=3, options=('A-star Search', 'Best First Graph Search', 'Brea…

Dropdown(description='Start city: ', options=('Arad', 'Bucharest', 'Craiova', 'Drobeta', 'Eforie', 'Fagaras', …

Dropdown(description='Goal city: ', index=5, options=('Arad', 'Bucharest', 'Craiova', 'Drobeta', 'Eforie', 'Fa…

interactive(children=(ToggleButton(value=False, description='visualize'), Output()), _dom_classes=('widget-int…

interactive(children=(IntSlider(value=0, description='iteration', max=1), Output()), _dom_classes=('widget-int…

### ผลลัพท์

| Algorithm | Total Interation |  Path | Total Cost |
| --- | --- | --- | --- |
| BF Tree Search | 742 | Arad -> Sibiu -> Fagaras -> Bucharest -> Urziceni -> Vaslui | 677 |
| BF Search | 42 | Arad -> Sibiu -> Fagaras -> Bucharest -> Urziceni -> Vaslui | 677 |
| DF Tree Search | *ไม่ทราบ (หน่วยความจำไม่พอ)* | *ไม่ทราบ (หน่วยความจำไม่พอ)* | *ไม่ทราบ* |
| DF Graph Search | 81 | Arad -> Timisoara -> Lugoj -> Mehadia -> Drobeta -> Craiova -> Pitesti -> Bucharest -> Urziceni -> Vaslui | 960 |
| Depth Limit Search | 81 | Arad -> Timisoara -> Lugoj -> Mehadia -> Drobeta -> Craiova -> Pitesti -> Bucharest -> Urziceni -> Vaslui | 960 |
| Uniform Cost Search | 54 | Arad -> Sibiu -> Rimnicu -> Pitesti -> Bucharest -> Urziceni -> Vaslui | 645 |
| Iterative Deepening Search | 81 | Arad -> Timisoara -> Lugoj -> Mehadia -> Drobeta -> Craiova -> Pitesti -> Bucharest -> Urziceni -> Vaslui | 960 |
| Greedy Best First Search | 26 | Arad -> Sibiu -> Fagaras -> Bucharest -> Urziceni -> Vaslui | 677 |
| A* Search | 43 | Arad -> Sibiu -> Rimnicu -> Pitesti -> Bucharest -> Urziceni -> Vaslui | 645 |
| Recursive Best First Search | 460 | Arad -> Sibiu -> Rimnicu -> Pitesti -> Bucharest -> Urziceni -> Vaslui | 645 |
 

### อธิบายเชิงเปรียบเทียบผลลัพธ์การค้นหาเส้นทาง Arad → Vaslui

**1) ความสมบูรณ์และประสิทธิภาพของ Tree Search เทียบกับ Graph Search**

จากตาราง DF Tree Search ไม่สามารถหาคำตอบได้เนื่องจากหน่วยความจำไม่เพียงพอ เพราะ Tree Search ไม่มีกลไกจดจำ state ที่เคยเยือนแล้ว (explored set) ทำให้เกิดการวนซ้ำ (infinite loop) ในกราฟที่มีวงจร เช่น Arad-Sibiu-Arad ไปเรื่อย ๆ จนพื้นที่หน่วยความจำหมดก่อนหาคำตอบพบ ในทางตรงกันข้าม DF Graph Search, Depth Limit Search และ Iterative Deepening Search ที่มีการจำกัดความลึกหรือมีการตรวจสอบ state ซ้ำ สามารถหาคำตอบได้สำเร็จด้วยจำนวนซ้ำ (iteration) เท่ากันคือ 81 ครั้ง และได้เส้นทางเดียวกันคือผ่าน Timisoara-Lugoj-Mehadia-Drobeta-Craiova-Pitesti ด้วยต้นทุนรวม 960 ซึ่งสะท้อนธรรมชาติของ DFS ที่มุ่งลึกก่อนกว้าง จึงมักไม่ใช่เส้นทางที่ประหยัดที่สุด

**2) Uninformed Search: BFS เทียบกับ UCS**

BF Tree Search และ BF Search ให้ผลลัพธ์เส้นทางเดียวกันคือ Arad-Sibiu-Fagaras-Bucharest-Urziceni-Vaslui ต้นทุน 677 แต่จำนวนซ้ำต่างกันมาก (742 เทียบกับ 42) เนื่องจาก Tree Search ไม่ตัดกิ่งที่ซ้ำออก ทำให้ต้องขยายโหนดจำนวนมากกว่า Graph Search หลายเท่า ในขณะที่ Uniform Cost Search ซึ่งพิจารณาต้นทุนสะสม (path cost) แทนจำนวนขั้น สามารถหาเส้นทางที่ประหยัดที่สุดจริง (645) ได้ด้วยจำนวนซ้ำเพียง 54 ครั้ง แสดงให้เห็นว่า BFS รับประกันเฉพาะจำนวนขั้นน้อยที่สุด ไม่ใช่ต้นทุนต่ำสุด หากต้นทุนแต่ละ edge ไม่เท่ากัน

**3) Informed Search: Greedy, A\* และ RBFS**

Greedy Best First Search ใช้เพียงค่า heuristic h(n) นำทาง จึงค้นหาได้เร็วที่สุดในกลุ่ม (26 iteration) แต่ได้เส้นทางที่ไม่เหมาะสมที่สุด (677) เพราะละเลยต้นทุนที่ผ่านมาแล้ว ในทางกลับกัน A* Search ซึ่งรวมทั้ง g(n) และ h(n) เข้าด้วยกัน (f = g + h) สามารถหาเส้นทางที่ประหยัดที่สุดได้ (645) เท่ากับ UCS แต่ใช้จำนวนซ้ำน้อยกว่าค่อนข้างมาก (43 เทียบกับ 54) สะท้อนว่า heuristic ที่ยอมรับได้ (admissible) ช่วยลดพื้นที่การค้นหาได้อย่างมีประสิทธิภาพโดยไม่เสียความเหมาะสมที่สุด

ส่วน Recursive Best First Search แม้จะให้คำตอบที่เหมาะสมที่สุดเช่นเดียวกับ A* (645) แต่ใช้จำนวนซ้ำสูงถึง 460 ครั้ง เนื่องจากกลไก backtracking ที่ต้องคำนวณ f-value ซ้ำเมื่อย้อนกลับขึ้นไปสำรวจกิ่งอื่น ทำให้สูญเสียข้อมูลบางส่วนที่เคยคำนวณไว้ (regenerate node) ต่างจาก A* ที่เก็บโหนดทั้งหมดไว้ใน memory เพื่อเปรียบเทียบ

**สรุป** จากทั้ง 9 อัลกอริทึมที่หาคำตอบได้ มีเพียง UCS, A* และ RBFS เท่านั้นที่ให้ต้นทุนต่ำสุด (645) ซึ่งยืนยันคุณสมบัติ optimality ของอัลกอริทึมทั้งสาม ในขณะที่ A* แสดงประสิทธิภาพด้านจำนวนการขยายโหนดที่ดีที่สุดในกลุ่มนี้ เนื่องจากใช้ heuristic ช่วยจำกัดขอบเขตการค้นหาโดยยังคงรับประกันคำตอบที่เหมาะสมที่สุดไว้ได้